In [1]:
"""
Full definition of a GPT Language Model, all of it in this single file.
References:
1) the official GPT-2 TensorFlow implementation released by OpenAI:
https://github.com/openai/gpt-2/blob/master/src/model.py
2) huggingface/transformers PyTorch implementation:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py
"""

import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

        self.key_cache = []
        self.value_cache = []

    def reset_cache(self):
        self.key_cache = []
        self.value_cache = []

    def forward(self, x, use_cache = False, key_padding_mask = None):
        # key_padding_mask: optional (B, T) bool, True = real token, False = pad.
        # T covers every key this call attends over (cached + new).
        B, T_temp, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)
        # T_temp is the number of new tokens: the prompt length during prefill, 1 during decode

        # calculate query, key, values for all heads in batch
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2) # each (B, T_temp, C)
        if use_cache:
          # append new k and v to KV cache, then attend over everything cached so far
          self.key_cache.append(k)
          self.value_cache.append(v)
          k = torch.cat(self.key_cache, dim=1) # (B, T, C)
          v = torch.cat(self.value_cache, dim=1) # (B, T, C)

        T = k.size(1) # total number of keys (cached + new)

        # move head forward to be the batch dim
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T_temp, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T_temp, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # build the attention mask (True = may attend).
        # is_causal aligns the mask top-left, which is only correct when T_temp == T.
        # Otherwise the new queries sit at the END of the key sequence, so align the mask bottom-right.
        if key_padding_mask is None and T_temp == T:
            attn_mask, is_causal = None, True
        else:
            attn_mask = torch.ones(T_temp, T, dtype=torch.bool, device=q.device).tril(diagonal=T - T_temp) # (T_temp, T)
            is_causal = False
            if key_padding_mask is not None:
                # no query may attend to a pad key
                attn_mask = attn_mask[None, None] & key_padding_mask[:, None, None, :].bool() # (B, 1, T_temp, T)
                # a pad query under left padding sees only pad keys -> fully masked row -> softmax NaN.
                # let those rows attend to everything; their outputs are garbage but never used.
                attn_mask = attn_mask | ~attn_mask.any(dim=-1, keepdim=True)

        # causal self-attention; Self-attend: (B, nh, T_temp, hs) x (B, nh, hs, T) -> (B, nh, T_temp, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            # (with a custom attn_mask PyTorch falls back from the Flash kernel to the mem-efficient/math one)
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0, is_causal=is_causal) # (B, nh, T_temp, hs)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1))) # (B, nh, T_temp, T)
            if attn_mask is None:
                attn_mask = self.bias[:,:,:T,:T] != 0
            att = att.masked_fill(~attn_mask, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T_temp, T) x (B, nh, T, hs) -> (B, nh, T_temp, hs)
        y = y.transpose(1, 2).contiguous().view(B, T_temp, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y # (B, T_temp, C)

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x, use_cache = False, key_padding_mask = None):
        x = x + self.attn(self.ln_1(x), use_cache, key_padding_mask)
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # with weight tying when using torch.compile() some warnings get generated:
        # "UserWarning: functional_call was passed multiple values for tied weights.
        # This behavior is deprecated and will be an error in future versions"
        # not 100% sure what this is, so far seems to be harmless. TODO investigate
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))


    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def reset_cache(self):
        for block in self.transformer.h:
            block.attn.reset_cache()

    def forward(self, idx, targets=None, use_cache = False, pos_value = 0, attention_mask = None):
        # pos_value: position of the first token in idx (only used with use_cache and no attention_mask)
        # attention_mask: optional (b, T_kv) tensor, 1 = real token, 0 = left pad. T_kv covers every key
        #   attended over: the cached tokens followed by the t tokens in idx (so T_kv == t without cache).
        device = idx.device
        b, t = idx.size()
        if attention_mask is None:
            assert pos_value + t <= self.config.block_size, f"Cannot forward sequence of length {pos_value + t}, block size is only {self.config.block_size}"
            pos = torch.arange(pos_value, pos_value + t, dtype=torch.long, device=device) # shape (t)
            key_padding_mask = None
        else:
            assert attention_mask.size(0) == b and attention_mask.size(1) >= t
            key_padding_mask = attention_mask.bool()
            # each row's positions count only its real tokens, so a left-padded row starts at position 0
            # just like it would unpadded. pads get position 0 (clamped); their outputs are ignored anyway.
            pos = (key_padding_mask.long().cumsum(dim=1) - 1).clamp(min=0)[:, -t:] # shape (b, t)
            assert pos.max().item() < self.config.block_size, f"Cannot forward sequence of length {pos.max().item() + 1}, block size is only {self.config.block_size}"

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd) or (b, t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x, use_cache, key_padding_mask)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss

    def crop_block_size(self, block_size):
        # model surgery to decrease the block size if necessary
        # e.g. we may load the GPT2 pretrained model checkpoint (block size 1024)
        # but want to use a smaller block size for some smaller, simpler model
        assert block_size <= self.config.block_size
        self.config.block_size = block_size
        self.transformer.wpe.weight = nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]

    @classmethod
    def from_pretrained(cls, model_type, override_args=None):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        override_args = override_args or {} # default to empty dict
        # only dropout can be overridden see more notes below
        assert all(k == 'dropout' for k in override_args)
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        print("forcing vocab_size=50257, block_size=1024, bias=True")
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        config_args['bias'] = True # always True for GPT model checkpoints
        # we can override the dropout rate, if desired
        if 'dropout' in override_args:
            print(f"overriding dropout rate to {override_args['dropout']}")
            config_args['dropout'] = override_args['dropout']
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer

    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        # express our flops throughput as ratio of A100 bfloat16 peak flops
        flops_achieved = flops_per_iter * (1.0/dt) # per second
        flops_promised = 312e12 # A100 GPU bfloat16 peak flops is 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, log_timing=True, attention_mask=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.

        For a batch of prompts with different lengths, LEFT-pad idx and pass attention_mask
        (shape (b,t), 1 = real token, 0 = pad). Left padding keeps every row's last real token in
        the final column, so all rows sample their next token from the same position. The returned
        idx still contains the pads; strip them with the mask.

        Timing stats are stored in self.last_timing (and printed if log_timing):
          ttft: time to first token (prefill + sampling the first token), seconds
          tpot: mean time per output token after the first (decode steps), seconds
          token_times: wall-clock latency of each generated token, seconds
        """
        def sync():
            # GPU kernels run asynchronously; wait for them so timings are real
            if idx.device.type == 'cuda':
                torch.cuda.synchronize(idx.device)
            elif idx.device.type == 'mps':
                torch.mps.synchronize()

        B, T = idx.size()
        assert max_new_tokens <= self.config.block_size, f"max_new_tokens ({max_new_tokens}) must be <= block_size ({self.config.block_size})"
        self.reset_cache() # clear K/V from any previous generate() call

        # crop the prompt so prompt + generated tokens fed back in fit in block_size.
        # the last sampled token is never fed back, so we need (T - start_index) + (max_new_tokens - 1) <= block_size
        start_index = max(0, T + max_new_tokens - 1 - self.config.block_size)

        sync()
        t_start = time.perf_counter()
        t_prev = t_start
        token_times = []

        # prefill: run the whole prompt through in one forward pass, filling the KV cache
        prompt = idx[:, start_index:]
        if attention_mask is not None:
            assert attention_mask.shape == idx.shape, "attention_mask must have the same shape as idx"
            attention_mask = attention_mask[:, start_index:].to(device=idx.device, dtype=torch.bool)
            assert attention_mask[:, -1].all(), "prompts must be LEFT-padded (last column must be real tokens)"
        logits, _ = self(prompt, use_cache = True, pos_value = 0, attention_mask = attention_mask)
        pos_value = prompt.size(1)

        # decode
        for i in range(max_new_tokens):
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)
            # token i is done once it has been sampled (token 0's time includes prefill)
            sync()
            t_now = time.perf_counter()
            token_times.append(t_now - t_prev)
            t_prev = t_now
            if i < max_new_tokens - 1:
                # forward only the new token; K/V for earlier tokens come from the cache
                if attention_mask is not None:
                    # the new token is real for every row
                    attention_mask = torch.cat((attention_mask, attention_mask.new_ones(B, 1)), dim=1)
                logits, _ = self(idx_next, use_cache = True, pos_value = pos_value, attention_mask = attention_mask)
                pos_value += 1

        ttft = token_times[0] if token_times else float('nan')
        decode_times = token_times[1:]
        tpot = sum(decode_times) / len(decode_times) if decode_times else float('nan')
        self.last_timing = dict(ttft=ttft, tpot=tpot, token_times=token_times,
                                prompt_len=prompt.size(1), new_tokens=len(token_times))
        if log_timing:
            print(f"prompt tokens: {prompt.size(1)}, new tokens: {len(token_times)}")
            print(f"time to first token: {ttft*1000:.2f} ms")
            print(f"time per output token: {tpot*1000:.2f} ms ({1/tpot if tpot > 0 else float('nan'):.1f} tok/s per sequence)")
            print(f"total: {(t_prev - t_start)*1000:.2f} ms")

        return idx

In [4]:
import time

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
model = GPT.from_pretrained('gpt2').to(device).eval()

# warm-up (not timed)
model.generate(torch.randint(0, 50257, (1, 512), device=device), 8, log_timing=False)

for B in [1, 4, 16]:
    print(f"\nTesting batch size {B}")
    idx = torch.randint(0, 50257, (B, 512), dtype=torch.long, device=device)
    model.generate(idx, 128)
    t = model.last_timing
    print(f"aggregate throughput: {B / t['tpot']:.1f} tok/s")

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
number of parameters: 123.65M


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Testing batch size 1
prompt tokens: 512, new tokens: 128
time to first token: 1545.15 ms
time per output token: 49.70 ms (20.1 tok/s per sequence)
total: 7857.46 ms
aggregate throughput: 20.1 tok/s

Testing batch size 4
prompt tokens: 512, new tokens: 128
time to first token: 5563.61 ms
time per output token: 137.57 ms (7.3 tok/s per sequence)
total: 23035.63 ms
aggregate throughput: 29.1 tok/s

Testing batch size 16
prompt tokens: 512, new tokens: 128
time to first token: 20453.35 ms
time per output token: 345.20 ms (2.9 tok/s per sequence)
total: 64294.22 ms
aggregate throughput: 46.3 tok/s


In [ ]:
# left-padded batch: prompts of different lengths in one generate() call
def left_pad(seqs, pad_id=50256):
    """list of 1-D LongTensors -> (idx, attention_mask), both (B, max_len), padded on the left"""
    T = max(len(s) for s in seqs)
    idx = torch.full((len(seqs), T), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), T), dtype=torch.long)
    for i, s in enumerate(seqs):
        idx[i, T - len(s):] = s
        mask[i, T - len(s):] = 1
    return idx, mask

torch.manual_seed(0)
lengths = [512, 300, 37, 5]
prompts = [torch.randint(0, 50257, (n,)) for n in lengths]
idx, mask = left_pad(prompts)
idx, mask = idx.to(device), mask.to(device)

# greedy (top_k=1) so padded and unpadded runs should pick the same tokens
out = model.generate(idx, 32, top_k=1, attention_mask=mask, log_timing=False)
for i, p in enumerate(prompts):
    ref = model.generate(p[None].to(device), 32, top_k=1, log_timing=False)
    match = torch.equal(out[i, -32:], ref[0, -32:])
    print(f"prompt len {lengths[i]:4d}: padded batch matches unpadded run: {match}")
